# LeetCode #152 — Maximum Product Subarray: giữ cả max và min

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def maxProduct(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Vì sao Kadane cho **tổng** (#53) không chạy được với **tích**: số âm đổi chiều lớn/nhỏ.
2. DP giữ **hai** trạng thái: tích lớn nhất **và nhỏ nhất** kết thúc tại `i`.
3. Cách nhìn khác: quét **tiền tố** và **hậu tố**, reset khi gặp 0.
4. Xử lý số 0 và số âm; `O(n)` thời gian, `O(1)` bộ nhớ.

# 1. Bài toán là gì?

Cho mảng số nguyên `nums`, tìm **mảng con liên tiếp khác rỗng** có **tích lớn nhất**, trả về tích đó.

Đề đảm bảo kết quả vừa số nguyên 32-bit.

**Constraints (theo đề):**

- `1 <= nums.length <= 2 * 10^4`
- `-10 <= nums[i] <= 10`
- Tích của mọi tiền tố hoặc hậu tố của `nums` đảm bảo vừa số nguyên 32-bit.

---

## Ví dụ

```text
[2, 3, -2, 4]      ->  6     ([2, 3])
[-2, 0, -1]        ->  0     ([0]; [-2, -1] không liên tiếp)
[-2, 3, -4]        ->  24    (cả mảng: hai số âm triệt tiêu)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 Áp dụng vào bài này

**Quy hoạch động** giống Kadane nhưng với tích: vì số âm đổi chiều lớn/nhỏ, trạng thái phải gồm **cả** tích lớn nhất và nhỏ nhất của đoạn con kết thúc tại `i`.

# 3. Nền tảng

In [3]:
from typing import List
import random, time
from math import prod

print("prod([2, 3, -2]) =", prod([2, 3, -2]))
print("max(…, …, …) nhận nhiều đối số:", max(3, -7, 5), min(3, -7, 5))

prod([2, 3, -2]) = -12
max(…, …, …) nhận nhiều đối số: 5 -7


## Kadane cho tổng (#53) — nhắc lại

`best_ending_here = max(x, best_ending_here + x)`: hoặc bắt đầu mới tại `x`, hoặc nối vào đoạn tốt nhất kết thúc ở phần tử trước.

Với **tích**, "nối vào đoạn có tích **lớn nhất**" không còn đủ: nếu `x < 0`, nối vào đoạn có tích **nhỏ nhất** (âm nhất) mới cho tích lớn nhất.

# 4. Baseline: thử mọi mảng con

Cố định `i`, nhân dần tới `j` — `O(n²)`.

In [4]:
def max_product_brute(nums: List[int]) -> int:
    best = nums[0]
    for i in range(len(nums)):
        p = 1
        for j in range(i, len(nums)):
            p *= nums[j]
            best = max(best, p)
    return best

print(max_product_brute([2, 3, -2, 4]), max_product_brute([-2, 0, -1]), max_product_brute([-2, 3, -4]))

6 0 24


# 5. Ý tưởng chính: DP giữ max và min

Gọi `hi[i]` / `lo[i]` = tích **lớn nhất** / **nhỏ nhất** của mảng con **kết thúc tại `i`**. Với `x = nums[i]`, mảng con kết thúc tại `i` là `[x]` hoặc (mảng con kết thúc tại `i − 1`) + `x`:

```text
hi[i] = max(x, hi[i-1] * x, lo[i-1] * x)
lo[i] = min(x, hi[i-1] * x, lo[i-1] * x)
đáp án = max hi[i]
```

Vì sao đủ? Tích `P * x` là hàm **đơn điệu** theo `P` (tăng nếu `x > 0`, giảm nếu `x < 0`, hằng nếu `x = 0`) → cực trị đạt tại một trong hai đầu `hi[i-1]` hoặc `lo[i-1]`.

**Cẩn thận:** tính `hi` và `lo` mới **cùng lúc** từ giá trị **cũ** (Python: gán bộ).

In [5]:
def max_product_dp(nums: List[int]) -> int:
    hi = lo = best = nums[0]
    for x in nums[1:]:
        hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)
        best = max(best, hi)
    return best

print(max_product_dp([2, 3, -2, 4]), max_product_dp([-2, 0, -1]), max_product_dp([-2, 3, -4]))

6 0 24


# 6. Mô phỏng

In [6]:
def trace_dp(nums):
    hi = lo = best = nums[0]
    print(f"{'i':>2} {'x':>3} | {'hi':>5} {'lo':>5} | best")
    print(f"{0:>2} {nums[0]:>3} | {hi:>5} {lo:>5} | {best}")
    for i, x in enumerate(nums[1:], 1):
        hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)
        best = max(best, hi)
        print(f"{i:>2} {x:>3} | {hi:>5} {lo:>5} | {best}")
    return best

_ = trace_dp([2, 3, -2, 4, -1])

 i   x |    hi    lo | best
 0   2 |     2     2 | 2
 1   3 |     6     3 | 6
 2  -2 |    -2   -12 | 6
 3   4 |     4   -48 | 6
 4  -1 |    48    -4 | 48


Ở `x = −2`: `lo` nhận `6 * −2 = −12` (âm nhất). Đến `x = −1`: `hi = −48 * −1 = 48` — tích của cả mảng. Nếu chỉ giữ `hi`, ta đã mất `−48`.

# 7. Cách nhìn khác: tiền tố và hậu tố

Xét một đoạn **không chứa 0**. Nếu số lượng số âm **chẵn** → tích cả đoạn dương, lớn nhất. Nếu **lẻ** → phải bỏ số âm **đầu tiên** (giữ hậu tố sau nó) hoặc số âm **cuối cùng** (giữ tiền tố trước nó).

Cả hai trường hợp đều là một **tiền tố** hoặc **hậu tố** của đoạn đó. Số `0` chia mảng thành các đoạn → khi tích chạy bằng 0, khởi động lại từ 1.

```text
best = max(tích tiền tố, tích hậu tố) qua mọi vị trí (reset khi gặp 0)
```

In [7]:
def max_product_prefix_suffix(nums: List[int]) -> int:
    n = len(nums)
    best = max(nums)                       # phòng trường hợp đáp án là một phần tử (ví dụ 0)
    pre = suf = 1
    for i in range(n):
        pre = (pre or 1) * nums[i]         # pre == 0 → bắt đầu lại
        suf = (suf or 1) * nums[n - 1 - i]
        best = max(best, pre, suf)
    return best

print(max_product_prefix_suffix([2, 3, -2, 4]), max_product_prefix_suffix([-2, 0, -1]), max_product_prefix_suffix([-2, 3, -4]))

6 0 24


# 8. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def maxProduct(self, nums: List[int]) -> int:
        hi = lo = best = nums[0]
        for x in nums[1:]:
            hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)
            best = max(best, hi)
        return best

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.maxProduct([2, 3, -2, 4]), 6)
check("Ví dụ 2", sol.maxProduct([-2, 0, -1]), 0)
check("Hai âm", sol.maxProduct([-2, 3, -4]), 24)
check("Một âm", sol.maxProduct([-3]), -3)
check("Toàn âm lẻ", sol.maxProduct([-1, -2, -3]), 6)
check("Có 0 giữa", sol.maxProduct([-2, 0, 3, -1, 4]), 4)

random.seed(152)
for _ in range(5000):
    nums = [random.randint(-4, 4) for _ in range(random.randint(1, 12))]
    exp = max_product_brute(nums)
    assert sol.maxProduct(nums) == exp == max_product_prefix_suffix(nums), nums
print("5000 mảng ngẫu nhiên: DP và tiền tố/hậu tố khớp brute force")

big = [random.choice([-1, 1, 0, 2, -2]) if random.random() < 0.02 else random.choice([1, -1]) for _ in range(2 * 10**4)]
t0 = time.time(); r = sol.maxProduct(big); t1 = time.time()
print(f"n = 2·10^4: {t1 - t0:.4f}s, kết quả = {r}")

Ví dụ 1          PASS  6
Ví dụ 2          PASS  0
Hai âm           PASS  24
Một âm           PASS  -3
Toàn âm lẻ       PASS  6
Có 0 giữa        PASS  4
5000 mảng ngẫu nhiên: DP và tiền tố/hậu tố khớp brute force


n = 2·10^4: 0.0077s, kết quả = 2048


# 9. Vì sao đúng? Độ phức tạp?

**DP:** mọi mảng con kết thúc tại `i` là `[x]` hoặc `S + [x]` với `S` kết thúc tại `i − 1`. Tập tích của các `S` nằm trong `[lo[i-1], hi[i-1]]` và chứa hai đầu mút; `P ↦ P·x` đơn điệu nên `max`/`min` của `P·x` đạt tại một đầu mút. Do đó công thức cho đúng `hi[i]`, `lo[i]`.

**Tiền tố/hậu tố:** đã lập luận ở mục 7 — mảng con tối ưu (nếu tích > 0 hoặc không có 0) là tiền tố hoặc hậu tố của một đoạn không chứa 0; còn nếu đáp án ≤ 0 thì `max(nums)` đã bao.

```text
Time:  O(n)
Space: O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Chỉ giữ max (Kadane nguyên bản)

In [10]:
def wrong_only_max(nums):
    hi = best = nums[0]
    for x in nums[1:]:
        hi = max(x, hi * x)
        best = max(best, hi)
    return best

print("Sai :", wrong_only_max([-2, 3, -4]), "| Đúng:", sol.maxProduct([-2, 3, -4]))

Sai :

 3 | Đúng: 24


## Lỗi 2: Cập nhật `hi` rồi dùng `hi` **mới** để tính `lo`

In [11]:
def wrong_sequential(nums):
    hi = lo = best = nums[0]
    for x in nums[1:]:
        hi = max(x, hi * x, lo * x)
        lo = min(x, hi * x, lo * x)                # SAI: hi đã bị ghi đè
        best = max(best, hi)
    return best

random.seed(7)
for _ in range(10000):
    nums = [random.randint(-3, 3) for _ in range(random.randint(1, 6))]
    if wrong_sequential(nums) != max_product_brute(nums):
        print("Phản ví dụ:", nums, "| sai:", wrong_sequential(nums), "| đúng:", max_product_brute(nums))
        break

Phản ví dụ: [-3, -1, 1, -3, 1] | sai: 9 | đúng: 3


Phải tính từ giá trị cũ: gán bộ `hi, lo = …, …` hoặc lưu `old_hi` trước.

## Lỗi 3: Khởi tạo `best = 0` (hoặc `hi = lo = 1`)

In [12]:
def wrong_init(nums):
    hi = lo = 1
    best = 0                                       # SAI: mảng con phải khác rỗng
    for x in nums:
        hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)
        best = max(best, hi)
    return best

print("Sai :", wrong_init([-3]), "| Đúng:", sol.maxProduct([-3]))

Sai : 0 | Đúng: -3


Mảng `[-3]` có đáp án `−3`, không phải `0`. Khởi tạo bằng `nums[0]`.

## Lỗi 4: Tiền tố/hậu tố không reset sau 0

Tích dính 0 mãi mãi → bỏ lỡ đoạn tốt phía sau số 0.

# 11. Mẹo nhận diện pattern

- **Mảng con liên tiếp, tối ưu tổng** → Kadane một trạng thái.
- **Tối ưu tích / phép toán đổi dấu** → giữ **cả max và min** (hai đầu mút).
- **Số âm chẵn/lẻ trong đoạn** → tiền tố/hậu tố.

Bài liên quan: **#53 Maximum Subarray**, **#1567 Maximum Length of Subarray With Positive Product**, **#628 Maximum Product of Three Numbers** (Easy), **#713 Subarray Product Less Than K**, **#918 Maximum Sum Circular Subarray**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng `hi`, `lo`, `best` cho `[-1, -2, -3, 0, 2, -5, -2]`.

## Bài 2 — Điền chỗ trống

In [13]:
def exercise_2_max_product(nums: List[int]) -> int:
    hi = lo = best = nums[0]
    for x in nums[1:]:
        # TODO: cập nhật hi, lo cùng lúc; cập nhật best
        pass
    return best


# assert exercise_2_max_product([2, 3, -2, 4]) == 6
# assert exercise_2_max_product([-2, 0, -1]) == 0
# assert exercise_2_max_product([-2, 3, -4]) == 24

## Bài 3 — Biến thể: Maximum Length of Subarray With Positive Product (LeetCode #1567)

Độ dài mảng con dài nhất có **tích dương**. Gợi ý DP: `pos` / `neg` = độ dài dài nhất mảng con kết thúc tại `i` có tích dương / âm.

In [14]:
def exercise_3_get_max_len(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_get_max_len([1, -2, -3, 4]) == 4
# assert exercise_3_get_max_len([0, 1, -2, -3, -4]) == 3
# assert exercise_3_get_max_len([-1, -2, -3, 0, 1]) == 2

## Bài 4 — Biến thể: trả về **chính mảng con** có tích lớn nhất

Trả về cặp chỉ số `(l, r)` (bao gồm) của một mảng con tối ưu. Gợi ý: cùng với `hi`, `lo` lưu chỉ số bắt đầu tương ứng.

In [15]:
def exercise_4_max_product_range(nums: List[int]) -> tuple:
    # TODO
    pass


# l4, r4 = exercise_4_max_product_range([2, 3, -2, 4]); assert prod([2, 3, -2, 4][l4:r4 + 1]) == 6
# l4b, r4b = exercise_4_max_product_range([-2, 3, -4]); assert (l4b, r4b) == (0, 2)

## Bài 5 — Mini challenge: Maximum Product of Three Numbers (LeetCode #628)

Không cần liên tiếp: chọn **3 số** bất kỳ có tích lớn nhất. Cùng ý tưởng "số âm nhỏ nhất cũng quan trọng": đáp án là `max(3 số lớn nhất, 2 số nhỏ nhất × số lớn nhất)`. Làm trong `O(n)` (không sắp xếp).

In [16]:
def exercise_5_max_product_three(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_max_product_three([1, 2, 3]) == 6
# assert exercise_5_max_product_three([1, 2, 3, 4]) == 24
# assert exercise_5_max_product_three([-10, -10, 1, 3, 2]) == 300
# assert exercise_5_max_product_three([-1, -2, -3]) == -6

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_dp` sau khi tự làm.

**Bài 3**
- Mức 1: `x > 0`: `pos += 1`, `neg = neg + 1 if neg else 0`.
- Mức 2: `x < 0`: `pos, neg = (neg + 1 if neg else 0), pos + 1`. `x == 0`: cả hai về 0.

**Bài 4**
- Mức 1: lưu `(hi, hi_start)` và `(lo, lo_start)`; ứng viên `[x]` có start `i`; ứng viên `hi*x` kế thừa `hi_start`, …

**Bài 5**
- Mức 1: duy trì 3 số lớn nhất và 2 số nhỏ nhất trong một lượt (hoặc `heapq.nlargest`/`nsmallest`).

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [17]:
print("Đáp án:", trace_dp([-1, -2, -3, 0, 2, -5, -2]))

 i   x |    hi    lo | best
 0  -1 |    -1    -1 | -1
 1  -2 |     2    -2 | 2
 2  -3 |     6    -6 | 6
 3   0 |     0     0 | 6
 4   2 |     2     0 | 6
 5  -5 |     0   -10 | 6
 6  -2 |    20    -2 | 20
Đáp án: 20


## Đáp án Bài 2

In [18]:
def answer_2_max_product(nums: List[int]) -> int:
    return Solution().maxProduct(nums)

assert answer_2_max_product([2, 3, -2, 4]) == 6
assert answer_2_max_product([-2, 0, -1]) == 0
assert answer_2_max_product([-2, 3, -4]) == 24
print("Bài 2 PASS (giống Solution ở mục 8)")

Bài 2 PASS (giống Solution ở mục 8)


## Đáp án Bài 3

In [19]:
def answer_3_get_max_len(nums: List[int]) -> int:
    pos = neg = best = 0
    for x in nums:
        if x > 0:
            pos, neg = pos + 1, (neg + 1 if neg else 0)
        elif x < 0:
            pos, neg = (neg + 1 if neg else 0), pos + 1
        else:
            pos = neg = 0
        best = max(best, pos)
    return best

def brute_max_len(nums):
    best = 0
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            if prod(nums[i:j + 1]) > 0:
                best = max(best, j - i + 1)
    return best

assert answer_3_get_max_len([1, -2, -3, 4]) == 4
assert answer_3_get_max_len([0, 1, -2, -3, -4]) == 3
assert answer_3_get_max_len([-1, -2, -3, 0, 1]) == 2
for _ in range(3000):
    a = [random.randint(-3, 3) for _ in range(random.randint(1, 10))]
    assert answer_3_get_max_len(a) == brute_max_len(a), a
print("Bài 3 PASS")

Bài 3 PASS

## Đáp án Bài 4

In [20]:
def answer_4_max_product_range(nums: List[int]) -> tuple:
    hi = lo = nums[0]
    hi_s = lo_s = 0
    best, best_l, best_r = nums[0], 0, 0
    for i in range(1, len(nums)):
        x = nums[i]
        cands = [(x, i), (hi * x, hi_s), (lo * x, lo_s)]
        hi, hi_s = max(cands)
        lo, lo_s = min(cands)
        if hi > best:
            best, best_l, best_r = hi, hi_s, i
    return best_l, best_r

l4, r4 = answer_4_max_product_range([2, 3, -2, 4]); assert prod([2, 3, -2, 4][l4:r4 + 1]) == 6
l4b, r4b = answer_4_max_product_range([-2, 3, -4]); assert (l4b, r4b) == (0, 2)
for _ in range(3000):
    a = [random.randint(-3, 3) for _ in range(random.randint(1, 10))]
    l, r = answer_4_max_product_range(a)
    assert 0 <= l <= r < len(a) and prod(a[l:r + 1]) == max_product_brute(a), a
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [21]:
def answer_5_max_product_three(nums: List[int]) -> int:
    m1 = m2 = m3 = float("-inf")         # 3 số lớn nhất: m1 >= m2 >= m3
    s1 = s2 = float("inf")               # 2 số nhỏ nhất: s1 <= s2
    for x in nums:
        if x > m1:
            m1, m2, m3 = x, m1, m2
        elif x > m2:
            m2, m3 = x, m2
        elif x > m3:
            m3 = x
        if x < s1:
            s1, s2 = x, s1
        elif x < s2:
            s2 = x
    return max(m1 * m2 * m3, s1 * s2 * m1)

from itertools import combinations
assert answer_5_max_product_three([1, 2, 3]) == 6
assert answer_5_max_product_three([1, 2, 3, 4]) == 24
assert answer_5_max_product_three([-10, -10, 1, 3, 2]) == 300
assert answer_5_max_product_three([-1, -2, -3]) == -6
for _ in range(3000):
    a = [random.randint(-6, 6) for _ in range(random.randint(3, 8))]
    assert answer_5_max_product_three(a) == max(prod(c) for c in combinations(a, 3)), a
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Tích đổi chiều khi gặp số âm → giữ **cả** `hi` và `lo` của mảng con kết thúc tại `i`: `hi, lo = max(x, hi·x, lo·x), min(x, hi·x, lo·x)` (cập nhật **cùng lúc**). Khởi tạo bằng `nums[0]`. Hoặc quét tiền tố + hậu tố, reset sau số 0. `O(n)`, `O(1)`.

---

## Checklist trước khi nộp

- [ ] Giữ cả max và min?
- [ ] Cập nhật `hi`, `lo` từ giá trị **cũ**?
- [ ] Khởi tạo bằng `nums[0]` (không phải 0 hay 1)?
- [ ] Số 0: công thức với ứng viên `x` tự reset?